# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import glob
import graphical_sampling as gs
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm
from package_sampling.utils import inclusion_probabilities

/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/usr/lib/R/library", R: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/usr/lib/R/library:/usr/lib/R/library:/usr/lib/R/

# Loading and Determining Population

In [3]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())
print(probs_dict.keys())
print(probs_dict['random'].keys())

dict_keys(['cluster', 'AggregatedPop1027', 'random', 'meuse', 'grid', 'RegularPop1000', 'swiss'])
dict_keys(['cluster', 'AggregatedPop1027', 'random', 'meuse', 'grid', 'RegularPop1000', 'swiss'])
dict_keys(['equal', 'unequal'])


In [4]:
N = 100
n = 16
coords = coords_dict['random']
probs = probs_dict['random']['equal']
modified_probs = inclusion_probabilities(probs, n=n)
pop = gs.Population(coords, modified_probs)

In [ ]:
# grid eq 4, 8, 16

# density	0.000000	0.000000
# moran	-0.456751	0.137355
# local_balance	0.411552	0.152782
# voronoi	0.166400	0.102328

# Building Initial Designs

In [5]:
orders = [
    # "lexico-yx",
    # "lexico-xy",
    # "random",
    # "angle_0",
    # "distance_0",
    # "projection",
    # "center",
    "spiral",
    # "max",
    # "snake",
    # "hilbert",
]

In [6]:
initial_designs = []
combines = list(itertools.product(orders, orders))
num_trials = 1
for units_order, zones_order in tqdm(combines, desc="Generating initial designs", total=len(combines), unit="orders"):
    best = None
    best_score = np.inf
    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(2, 2),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))

    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=4,
            zone_builder='cluster',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(1, 1),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))

    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=3,
            zone_builder='cluster',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))


Generating initial designs:   0%|          | 0/1 [00:00<?, ?orders/s]

/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/us

In [7]:
for design in initial_designs:
    print(design.kmeans.all_samples.shape, design.kmeans.expected_moran_score())

(46, 16) -0.48011207191836563
(22, 16) -0.4933424372685568
(22, 16) -0.4933424372685568


# Run

In [9]:
moran_criteria = gs.criteria.MoranCriteria()

In [30]:
initial_designs = [astar.best_design]

In [10]:
astar = gs.search.AStar(
    initial_designs,
    moran_criteria
)

best initial criteria value -0.4933424372685568


In [32]:
astar.best_design.kmeans.score_summary_df()

,expected,std
measure,,
density,0.026160,0.090017
moran,-0.426376,0.057762
local_balance,0.179846,0.036052
voronoi,0.109324,0.024679


In [33]:
astar.run(
    max_iterations = 100,
    num_new_nodes = 20,
    max_open_set_size = 1000,

    n_clusters_to_change_order_zone = 3,
    n_changes_in_order_of_zones = 1,

    n_clusters_to_change_order_units = 2,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=8
)


parent node: -0.4263757184568576


/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/us

child node: -0.4090962118180046
child node: -0.42194004182250916
child node: -0.4130323616202906
child node: -0.39027997833779454
child node: -0.41572088245542094
child node: -0.4122387650675112
child node: -0.3746786552611437
child node: -0.413982825370247
child node: -0.37550145703896165
child node: -0.3730807292053521
child node: -0.37523532445439384
child node: -0.3721749924127788
child node: -0.3454240521884859
child node: -0.3974559926572252
child node: -0.42522977485350094
child node: -0.42144854336824833
child node: -0.3872367258364171
child node: -0.4124122636867605
child node: -0.36778002405536064
child node: -0.41144320041316623

parent node: -0.42522977485350094
child node: -0.4069921677301949
child node: -0.4146804800713497
child node: -0.41418082242166465
child node: -0.40095071829173284
child node: -0.379631735518535
child node: -0.3720204471110664
child node: -0.39565051812409135
child node: -0.4185060639816816
child node: -0.34743668452075177
child node: -0.40437209548

KeyboardInterrupt: 

In [11]:
astar.best_design.kmeans.all_samples_probs.sum()

np.float64(1.0)

In [12]:
astar.best_design.kmeans.fips

array([0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16, 0.16,
       0.16])

In [23]:
astar.best_design.kmeans.all_samples_probs

array([0.03965747, 0.03292318, 0.03192551, 0.03192551, 0.03225806,
       0.03192551, 0.01696043, 0.01646159, 0.03117725, 0.0152145 ,
       0.03192551, 0.03092784, 0.01646159, 0.01596275, 0.01546392,
       0.01571333, 0.01571334, 0.01596275, 0.01671101, 0.01496508,
       0.00798138, 0.01596275, 0.01696043, 0.00848021, 0.01546392,
       0.00748254, 0.01571334, 0.01496508, 0.01596275, 0.00798138,
       0.0152145 , 0.00099767, 0.01546392, 0.01496508, 0.0152145 ,
       0.01546392, 0.01596275, 0.00848021, 0.01546392, 0.00648487,
       0.00848021, 0.00748254, 0.00798138, 0.00848021, 0.01596275,
       0.01446625, 0.01646159, 0.00748254, 0.00748254, 0.00024942,
       0.00049884, 0.00099767, 0.00074825, 0.00748254, 0.01571333,
       0.00099767, 0.00099767, 0.01496508, 0.00074825, 0.00024942,
       0.00074825, 0.00848021, 0.00748254, 0.00024942, 0.00074825,
       0.00024942, 0.0082308 , 0.00074825, 0.00099767, 0.00723312,
       0.00823079, 0.00773196, 0.0082308 , 0.00099767, 0.00848